In [10]:
import os, sys, re, json, csv, time, importlib, subprocess

INTERACTIVE = False        # True -> live chatbot loop at the very end (type 'exit' to stop)
USE_EMBEDDINGS = os.environ.get("LAB6_SKIP_EMBED") != "1"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUT_DIR = "lab6_nlp"


def _ensure(mod, pkg):
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("numpy", "numpy")
_ensure("sklearn", "scikit-learn")
_ensure("docx", "python-docx")
if USE_EMBEDDINGS:
    try:
        _ensure("sentence_transformers", "sentence-transformers")
    except Exception as e:  # pragma: no cover
        print("Could not install sentence-transformers:", e)

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(OUT_DIR)
print("Working folder:", os.getcwd())


def H(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)


# ----------------------------------------------------------------------------
# Embedding model (loaded once, load time measured separately)
# ----------------------------------------------------------------------------
_MODEL = None
EMBEDDING_OK = USE_EMBEDDINGS
MODEL_LOAD_SECONDS = None


def get_model():
    """Load the encoder once. Returns None if it cannot be loaded."""
    global _MODEL, EMBEDDING_OK, MODEL_LOAD_SECONDS
    if _MODEL is None and EMBEDDING_OK:
        try:
            t = time.perf_counter()
            from sentence_transformers import SentenceTransformer
            _MODEL = SentenceTransformer(MODEL_NAME, device="cpu")
            MODEL_LOAD_SECONDS = time.perf_counter() - t
            print(f"[model] loaded {MODEL_NAME} in {MODEL_LOAD_SECONDS:.2f}s, "
                  f"max_seq_length={_MODEL.max_seq_length}")
        except Exception as e:
            EMBEDDING_OK = False
            print("[model] COULD NOT LOAD ENCODER:", repr(e))
            print("[model] Embedding parts will be skipped; record this limitation in the report.")
    return _MODEL


# =============================================================================
# SHARED HELPER  (lab6_search.py)
# =============================================================================
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            self.model = get_model()
            if self.model is None:
                raise RuntimeError("Encoder unavailable")
            self.check_length(texts)
            self.matrix = self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])


def compare_backends(records, query, k=3, filters=None, label=""):
    """Experiment helper: run TF-IDF and (if available) embedding on the same query."""
    print(f"\n--- experiment {label}: query={query!r} filters={filters}")
    for backend in ["tfidf", "embedding"]:
        if backend == "embedding" and get_model() is None:
            print("  [embedding skipped: encoder unavailable]")
            continue
        hits = Retriever(records, backend).search(query, k=k, filters=filters)
        print(f"  {backend:9s} ->", [(r["id"], round(s, 3)) for r, s in hits] or "no matches")


# =============================================================================
# SCENARIOS 1-5
# =============================================================================
def run_scenarios():
    H("SCENARIO 1 - University handbook search")
    rec1 = [
        {"id": "assessment:late", "source": "handbook.txt",
         "text": "Late submission is normally penalized. Students with illness "
                 "may request an extension with medical evidence before the deadline."},
        {"id": "assessment:format", "source": "handbook.txt",
         "text": "Submit the assignment as a PDF through the learning portal."}]
    hits = Retriever(rec1).search("illness medical evidence late submission", k=1)
    show_hits(hits)
    assert hits[0][0]["id"] == "assessment:late"
    compare_backends(rec1, "I was sick; can I hand it in tomorrow?", k=2, label="S1 paraphrase")

    H("SCENARIO 2 - Customer support knowledge base")
    rec2 = [
        {"id": "returns:damage", "region": "PK",
         "text": "For a damaged parcel, contact support within 48 hours "
                 "and attach photographs of the item and packaging."},
        {"id": "returns:change", "region": "PK",
         "text": "For a change of mind, return unused items within 14 days."},
        {"id": "returns:damage:other", "region": "OTHER",
         "text": "Damaged parcel reports in the other region have a 72 hour window."}]
    hits = Retriever(rec2).search("damaged parcel photographs", k=1, filters={"region": "PK"})
    show_hits(hits)
    assert hits[0][0]["id"] == "returns:damage" and "48 hours" in hits[0][0]["text"]
    compare_backends(rec2, "damaged parcel photographs", k=3, label="S2 NO filter")
    compare_backends(rec2, "My parcel arrived broken", k=3, label="S2 paraphrase")

    H("SCENARIO 3 - Software documentation search")
    rec3 = [
        {"id": "auth:v2", "version": "v2",
         "text": "Renew an expired API token with POST /v2/token/refresh. "
                 "Send the refresh_token field; never print its value in logs."},
        {"id": "auth:v1", "version": "v1",
         "text": "Renew an expired API token with POST /v1/auth/renew."}]
    eng = Retriever(rec3)
    hits = eng.search("renew expired API token", k=1, filters={"version": "v2"})
    show_hits(hits)
    assert hits[0][0]["id"] == "auth:v2"
    assert "POST /v2/token/refresh" in hits[0][0]["text"] and "refresh_token" in hits[0][0]["text"]
    print("filter v1 ->", [r["id"] for r, _ in eng.search("renew expired API token", 1, {"version": "v1"})])
    nohit = eng.search("renew expired API token", 1, {"version": "v9"})
    print("filter v9 (nonexistent) ->", nohit)
    assert nohit == []

    H("SCENARIO 4 - Workplace policy retrieval")
    rec4 = [
        {"id": "remote:staff", "audience": "employee",
         "text": "Employees may request remote work with manager approval."},
        {"id": "remote:contractor", "audience": "contractor",
         "text": "Contractors may work remotely only when the signed contract "
                 "explicitly permits remote work."}]
    hits = Retriever(rec4).search("remote work", k=1, filters={"audience": "contractor"})
    show_hits(hits)
    assert hits[0][0]["id"] == "remote:contractor" and "contract" in hits[0][0]["text"]
    compare_backends(rec4, "remote work", k=2, label="S4 NO filter (risk: employee rule may win)")
    print("NOTE: metadata filtering here is NOT an access-control/security system.")

    H("SCENARIO 5 - Equipment troubleshooting")
    rec5 = [
        {"id": "printer:blank", "device": "printer",
         "text": "For blank printer pages: 1. Check ink level. "
                 "2. Run the printer cleaning utility. "
                 "3. Print a test page. Follow the device manual."},
        {"id": "scanner:blank", "device": "scanner",
         "text": "For a blank scanner image, check document placement "
                 "and select the correct scanning source."}]
    hits = Retriever(rec5).search("blank pages", k=1, filters={"device": "printer"})
    show_hits(hits)
    r = hits[0][0]
    assert r["id"] == "printer:blank" and all(s in r["text"] for s in ["1.", "2.", "3."])
    compare_backends(rec5, "blank pages", k=2, label="S5 no filter")
    compare_backends(rec5, "blank image", k=2, label="S5 no filter")


# =============================================================================
# GUIDED EXAMPLES  (6.1 - 6.7 except chunks which need Task 2 functions)
# =============================================================================
def guided_tokens():
    H("6.1 guided_tokens.py")
    text = "I can't submit CS4944 late!"
    whitespace = text.split()
    words = re.findall(r"\b\w+\b", text)
    print("Whitespace:", whitespace, len(whitespace))
    print("Word-like:", words, len(words))
    print("Original:", text)
    for extra in ["help@example.com", "", "یہ ایک مثال ہے"]:
        print(repr(extra), "ws:", extra.split(), "| regex:", re.findall(r"\b\w+\b", extra))


def guided_embeddings():
    H("6.2 guided_embeddings.py")
    model = get_model()
    if model is None:
        print("skipped (encoder unavailable)")
        return
    passages = ["Reset a forgotten password using the recovery link.",
                "Submit assignments through the learning portal.",
                "Check ink levels when a printer produces blank pages."]
    C = model.encode(passages, normalize_embeddings=True)
    for label, qtext in [("paraphrase", "How can I recover my login?"),
                         ("NEGATED (experiment)", "I do not need a password reset")]:
        Q = model.encode([qtext], normalize_embeddings=True)
        scores = (Q @ C.T)[0]
        print(f"\nQuery [{label}]: {qtext!r}   embedding shape: {C.shape}")
        for i in np.argsort(-scores):
            print(" ", i, round(float(scores[i]), 3), passages[i])
    assert np.allclose(np.linalg.norm(C, axis=1), 1, atol=1e-5)
    print("\nObservation: the negated query still favours the password passage - the encoder")
    print("captures topic similarity, not logical negation. Score != correctness.")


def guided_limits():
    H("6.4 guided_limits.py")
    model = get_model()
    if model is None:
        print("skipped (encoder unavailable)")
        return
    for label, text in [("short", "Reset my password."), ("long", "password " * 300)]:
        ids = model.tokenizer(text, truncation=False)["input_ids"]
        print(label, "tokens:", len(ids), "limit:", model.max_seq_length,
              "fits:", len(ids) <= model.max_seq_length)
    for text in ["CS4944", "help@example.com", "POST /v2/token/refresh", "refresh_token"]:
        print(f"{text!r}: whitespace={len(text.split())} model_tokens="
              f"{len(model.tokenizer(text, truncation=False)['input_ids'])} (incl. special tokens)")


def retrieval_metrics(rankings, relevant):
    hits, rr = [], []
    for ids, good in zip(rankings, relevant):
        ranks = [r for r, chunk_id in enumerate(ids[:3], 1) if chunk_id in good]
        hits.append(bool(ranks))
        rr.append(1 / ranks[0] if ranks else 0)
    return sum(hits) / len(hits), sum(rr) / len(rr)


def guided_metrics():
    H("6.5 guided_metrics.py")
    rankings = [["a", "b", "c"], ["c", "a", "b"], ["x", "y", "z"]]
    relevant = [{"a"}, {"a"}, {"a"}]
    hit, mrr = retrieval_metrics(rankings, relevant)
    print(f"Hit@3={hit:.3f}; MRR@3={mrr:.3f}")
    assert abs(hit - 2 / 3) < 1e-9 and abs(mrr - 0.5) < 1e-9
    h2, m2 = retrieval_metrics([["x", "y", "z", "a"]], [{"a"}])
    print("relevant at rank 4 ->", h2, m2, "(must be a miss at cutoff 3)")
    assert h2 == 0 and m2 == 0


# =============================================================================
# TASK 2 CORE: CHUNKING MODULE  (task2_chunks.py)
# =============================================================================
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    # Teaching splitter: punctuation followed by whitespace.
    cuts = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:                      # fallback: long sentence -> word windows
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    # trim trailing whitespace so that offsets point at visible text
    trimmed = []
    for a, b in out:
        seg = text[a:b]
        trimmed.append((a, b - (len(seg) - len(seg.rstrip()))))
    return trimmed


def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = (sentence_chunks(text) if strategy == "sentence"
                 else windows(text, 40, int(strategy)))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id, start=a, end=b, text=text[a:b]))
    return result


def chunk_self_tests():
    H("TASK 2 - boundary tests for chunking")
    assert windows("") == [] and sentence_chunks("") == []
    print("empty text -> no chunks: OK")
    assert windows("short text") == [(0, 10)]
    _s = "Short doc. Under budget."
    assert sentence_chunks(_s) == [(0, len(_s))]
    print("document shorter than budget -> exactly one chunk: OK")
    for bad in [(2, 2), (2, 5), (0, 0), (5, -1)]:
        try:
            windows("a b c", *bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"invalid size/overlap {bad} accepted")
    print("invalid overlap/size -> ValueError: OK")
    long = " ".join(["word"] * 95) + "."
    sc = sentence_chunks(long)
    print("95-word sentence ->", len(sc), "fallback windows", [len(long[a:b].split()) for a, b in sc])
    assert all(len(long[a:b].split()) <= 40 for a, b in sc) and len(sc) == 3
    docs = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ["0", "10", "sentence"]:
        for c in make_chunks(docs, strategy):
            assert c["text"] == docs[c["doc"]][c["start"]:c["end"]]
    print("offset check text == source[start:end]: OK")

    H("6.3 guided_chunks.py")
    text = ("Late work is penalized. Illness may qualify for an extension "
            "when medical evidence is provided. Ask the coordinator first.")
    wspans, sspans = windows(text, size=12, overlap=4), sentence_chunks(text, size=12)
    for name, spans in [("word windows", wspans), ("sentence aware", sspans)]:
        print(name)
        for start, end in spans:
            print(" ", start, end, repr(text[start:end]))
            assert 0 <= start < end <= len(text)
    assert len(wspans) == 2 and len(sspans) == 3
    print("20-word budget experiment:", [(a, b) for a, b in sentence_chunks(text, 20)])

    print("\nWhere sentence splitting FAILS (abbreviations):")
    t = "Dr. Imran approved approx. two days. Contact the office."
    for a, b in sentence_chunks(t, size=3):
        print("  ", repr(t[a:b]))
    print("  -> the regex cuts after 'Dr.' and 'approx.' although they do not end sentences;")
    print("     a rule or evidence sentence can be separated from its condition.")


# =============================================================================
# SYNTHETIC CORPUS + LABELLED QUERIES  (create_dataset.py, expanded)
# =============================================================================
DOCS = {
    "late": (
        "Assessment Policy, version 2024. Students must submit every assignment through the "
        "learning portal before the stated deadline. Late submission is normally penalized by "
        "ten percent of the marks for each calendar day after the deadline, and work more than "
        "five days late is not accepted. Dr. Imran, the programme coordinator, reviews all "
        "extension cases personally and records the outcome in the student file, which is why "
        "requests sent to other staff are not processed. Illness extensions require medical "
        "evidence before the deadline, and the certificate must be issued by a registered doctor "
        "and show the dates of illness. A student who cannot obtain evidence in time should "
        "contact the coordinator immediately and explain the situation in writing, although "
        "approval is not guaranteed. Extensions granted for illness are limited to seven days "
        "unless the examination committee approves a longer period for serious cases. Group work "
        "follows the same rules, and every member must be named in the submission. Plagiarism "
        "checks are applied to every submission, including those that are late or resubmitted "
        "after an extension. Students should keep a copy of the receipt generated by the portal."),
    "late_archived": (
        "Assessment Policy, version 2019 (archived). Assignments were submitted by email to the "
        "course instructor until the learning portal was introduced. Late submission was not "
        "penalized during the first three calendar days, and after that a flat penalty of twenty "
        "percent applied to the whole assignment. Illness extensions require only a written note "
        "from a parent or guardian and no medical evidence is needed. The instructor could approve "
        "an extension of up to fourteen days without consulting the coordinator, and verbal "
        "approval was considered sufficient when the student kept the email thread as proof. This "
        "version was replaced because several cases could not be verified later, and departments "
        "were asked to stop using it after the 2020 revision. It is kept in the archive only so "
        "that old appeals can be reviewed against the rules that applied at that time. Students "
        "must never rely on this archived text when planning a current submission, and staff "
        "should always cite the current version when answering questions from students."),
    "password": (
        "IT Help Desk Guide: Account Access. Every student receives a university account that is "
        "used for the learning portal, the library catalogue and the campus wireless network. "
        "Accounts are locked automatically after five failed sign-in attempts, and the lock is "
        "removed after thirty minutes without further attempts. Reset a forgotten password using "
        "the recovery link on the login page. The link is sent to the personal email address "
        "registered during admission, so students should update that address whenever it changes. "
        "Recovery codes are shown only once when two-step verification is enabled, and they must "
        "be stored somewhere safe because the help desk cannot display them again. Staff will "
        "never ask for a password or a recovery code by telephone, chat or email, and any message "
        "that asks for one should be reported to the security team immediately. Students who no "
        "longer have access to their registered email address must visit the help desk in person "
        "with their student card so that identity can be checked before the account is restored. "
        "The help desk is open on weekdays from nine in the morning until four in the afternoon."),
    "assignment": (
        "Coursework Submission Guide. Assignments must be uploaded through the learning portal as "
        "a single PDF file, and files sent by email, messaging applications or removable drives "
        "are not accepted for marking under any circumstances. The portal records the exact time "
        "of each upload, and when a student uploads several versions before the deadline only the "
        "most recent version is marked, which means that every upload should be checked carefully "
        "because an older and better version will not be recovered afterwards by the lecturer or "
        "the coordinator. File names should contain the course code and the student number, for "
        "example CS4944_42365.pdf. The maximum file size is twenty megabytes, so scanned pages "
        "should be compressed before uploading. Source code must be included as an appendix or as "
        "a link to a repository that stays accessible until the final grades are published. "
        "Students who see an error message during upload should take a screenshot and contact the "
        "help desk before the deadline passes. The receipt email from the portal is the only "
        "accepted proof that a submission was made."),
    "contractor": (
        "Remote Work Policy: Contractors. This section applies to external contractors engaged "
        "through an agency or on a fixed-term agreement. Contractors can work remotely only if the "
        "signed contract permits it. Where the contract is silent, the contractor must work from "
        "the assigned office location and must not connect to internal systems from personal "
        "devices. Employee rules do not apply to contractors, and approval from a line manager "
        "cannot replace a clause in the contract. Any change to the working location must be "
        "agreed in writing by the contracts office before it starts. Contractors who are permitted "
        "to work remotely must use the university virtual private network and keep project files "
        "on approved storage only. Equipment supplied by the university remains university "
        "property and must be returned on the last day of the contract. Questions about the "
        "interpretation of a contract clause should be sent to the contracts office, and managers "
        "should not give verbal permission in advance. This policy was last reviewed in March by "
        "the human resources department."),
    "employee_remote": (
        "Remote Work Policy: Employees. This section applies to permanent staff who have completed "
        "their probation period. Employees may request remote work for up to two days per week "
        "with the approval of their line manager. Approval is recorded in the staff portal and "
        "reviewed every six months, and the manager may withdraw it when the needs of the "
        "department change. Employees who work remotely must be reachable during core hours, which "
        "run from ten in the morning until three in the afternoon. They must use the university "
        "virtual private network and keep project files on approved storage only. Equipment "
        "supplied by the university remains university property, and any damage must be reported "
        "to the IT help desk within two working days. Staff on probation are not eligible for "
        "remote work unless the head of department grants a written exception. Contractors are "
        "covered by a separate section of this policy and should not use this section to justify "
        "remote work. This policy was last reviewed in March by the human resources department."),
    "printer": (
        "Equipment Guide: Printers and Scanners. Before any troubleshooting, switch off the device "
        "and disconnect the power cable if you need to open a cover, because internal parts can be "
        "hot or carry a charge. For blank printer pages, check ink levels and run the cleaning "
        "utility. If the pages are still blank after cleaning, print a test page and wait approx. "
        "two minutes before repeating the cleaning cycle, e.g. when the nozzles are clogged after "
        "a long period of inactivity. Replace the cartridge only with a model listed in the device "
        "manual, and do not refill cartridges because refilled ink can damage the print head and "
        "cancel the warranty. A scanner that produces a blank image is a different problem, so "
        "check the document placement and select the correct scanning source in the scanner "
        "settings. Paper jams should be cleared gently by pulling the sheet in the direction of "
        "travel. Faults that remain after these steps must be reported to the IT help desk with "
        "the device model, the serial number and a description of what was tried. Maintenance "
        "staff will visit the department on Thursdays."),
}

EVIDENCE = {
    "late": "Illness extensions require medical evidence before the deadline, and the certificate "
            "must be issued by a registered doctor and show the dates of illness.",
    "password": "Reset a forgotten password using the recovery link on the login page.",
    "assignment": "Assignments must be uploaded through the learning portal as a single PDF file",
    "contractor": "Contractors can work remotely only if the signed contract permits it.",
    "printer": "For blank printer pages, check ink levels and run the cleaning utility.",
}

QUESTIONS = [  # (query, doc, split) - labelled BEFORE looking at any ranking
    ("What is required for illness extensions?", "late", "dev"),
    ("How do I reset a forgotten password?", "password", "dev"),
    ("Where do I upload assignments?", "assignment", "dev"),
    ("Can contractors work remotely?", "contractor", "dev"),
    ("What are tomorrow's flight prices?", None, "dev"),
    ("What do I do for blank printer pages?", "printer", "test"),
    ("How can I recover my login access?", "password", "test"),
    ("How should I hand in my coursework?", "assignment", "test"),
    ("What proof is needed when I was ill?", "late", "test"),
    ("Who won the football match?", None, "test"),
]
QUERIES = [dict(query=q, doc=d, evidence=EVIDENCE[d] if d else None, split=s) for q, d, s in QUESTIONS]


def write_dataset():
    H("DATASET - corpus.json / queries.json")
    for k, t in DOCS.items():
        n = len(t.split())
        print(f"{k:16s} {n:4d} words", "" if 150 <= n <= 250 else "  <-- outside 150-250!")
    for q in QUERIES:
        assert q["split"] in {"dev", "test"}
        if q["doc"] is not None:
            assert q["evidence"] in DOCS[q["doc"]], f"evidence not an exact substring: {q['doc']}"
    with open("corpus.json", "w", encoding="utf-8") as f:
        json.dump(DOCS, f, ensure_ascii=False, indent=2)
    with open("queries.json", "w", encoding="utf-8") as f:
        json.dump(QUERIES, f, ensure_ascii=False, indent=2)
    print("Wrote corpus.json and queries.json;", sum(q['doc'] is not None for q in QUERIES),
          "answerable +", sum(q['doc'] is None for q in QUERIES), "unanswerable queries")


# =============================================================================
# TASK 1 - tokenization + FAQ chatbot
# =============================================================================
TOKEN_ROWS = []


def task1_tokens():
    H("TASK 1 (Beginner) - whitespace vs Unicode word-like regex vs model sub-words")
    samples = ["AI, search!", "I can't log in.", "help@example.com", "CS4944",
               "", "یہ ایک مثال ہے", "POST /v2/token/refresh", "Don’t panic — mail me@uni.edu.pk"]
    model = get_model()
    for text in samples:
        ws = text.split()
        words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
        row = dict(text=text, ws=ws, words=words, sub=None)
        print(f"\nOriginal : {text!r}")
        print(f"whitespace ({len(ws)}): {ws}")
        print(f"word-like  ({len(words)}): {words}")
        if model is not None and text in ("I can't log in.", "CS4944", "help@example.com"):
            row["sub"] = model.tokenizer.tokenize(text)
            print(f"model sub-words ({len(row['sub'])}): {row['sub']}")
        TOKEN_ROWS.append(row)
    assert re.findall(r"\b\w+\b", "") == []
    print("""
EXPLANATION
 1) Punctuation: split() keeps it attached ("search!" is one unit) while the regex drops it,
    so the counts/tokens differ for 'AI, search!'.
 2) Contraction / identifiers: the regex splits "can't" into "can" + "t" and the email into
    help / example / com, whereas split() keeps them whole. Sub-word tokenizers split again
    in their own way, so none of the three lists is equal.
 NORMALIZATION RISK: lowercasing, stripping punctuation or splitting blindly can destroy
    meaning: "can't" -> "can t" loses the negation, "POST /v2/token/refresh" loses the case-
    sensitive path, "CS4944" and e-mail addresses are no longer recognisable identifiers.
    Keep the original text; use any normalised copy only for searching.
 The \\w regex handles Urdu letters but is NOT a complete multilingual tokenizer (no
    morphology, no handling of joiners/diacritics, no word segmentation for e.g. Chinese).""")


FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.", "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.", "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.", "source": "handbook.txt"},
    # two FAQs added for Task 1 (each with a source)
    {"id": "faq:wifi", "question": "How do I connect to the campus Wi-Fi?",
     "answer": "Select the campus network and sign in with your university account.",
     "source": "it_help.txt"},
    {"id": "faq:library", "question": "When is the library open?",
     "answer": "The library is open on weekdays from 8 am to 8 pm.", "source": "library.txt"},
]


def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"] + " " + r["answer"]) for r in FAQ]
    return Retriever(records, backend)


def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    for r in FAQ:  # only exact canonical FAQ text is answered automatically
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)


FAQ_TRANSCRIPT = []


def task1_chatbot():
    H("TASK 1 - FAQ chatbot (TF-IDF) transcript + checks")
    engine = build_bot("tfidf")
    for q, label in [("How do I reset my password?", "exact answer"),
                     ("password recovery", "suggestion (non-exact)"),
                     ("When is the library open?", "exact answer (added FAQ)"),
                     ("", "empty input"),
                     ("volcano geology", "unsupported input")]:
        a = reply(engine, q)
        FAQ_TRANSCRIPT.append((label, q, a))
        print(f"[{label}]\nYou: {q}\nBot: {a}\n")
    assert "it_help.txt" in reply(engine, "How do I reset my password?")
    assert reply(engine, "") == "Please enter a question."
    assert "No matching FAQ" in reply(engine, "volcano geology")
    assert "Possible matches" in reply(engine, "password recovery")
    print("FAQ behaviour checks passed")
    print("""WHY CONFIRMATION? A similarity score only measures resemblance, not whether the
stored answer truly answers the user's question (negation, two topics in one question,
unrelated queries still get nonzero semantic scores). The bot therefore auto-answers only
exact canonical questions and shows candidates for the user to confirm.""")
    if get_model() is not None:
        eb = build_bot("embedding")
        print("--- embedding backend on non-exact queries ---")
        for q in ["I forgot my login details", "Can I hand it in late because I was sick?",
                  "How do I reset my password and when is the library open?", "volcano geology"]:
            print("You:", q, "\nBot:", reply(eb, q), "\n")


def guided_suggestions():
    H("6.7 guided_suggestions.py")
    for q in ["assignment upload", "forgot login"]:
        for backend in ["tfidf", "embedding"]:
            if backend == "embedding" and get_model() is None:
                continue
            eng = build_bot(backend)
            print(f"[{backend}] query={q!r}")
            for record, score in eng.search(q, k=3):
                if backend == "embedding" or score > 0:   # >0 rule is lexical only
                    print("   ", record["question"], f"({score:.3f})")


# =============================================================================
# TASK 2 - chunk statistics + semantic suggestions on chunks
# =============================================================================
def chunk_stats(docs, strategy):
    ch = make_chunks(docs, strategy)
    sizes = [len(c["text"].split()) for c in ch]
    orig = sum(len(t.split()) for t in docs.values())
    # does each labelled rule+exception survive intact inside ONE chunk?
    surv = {d: any(c["doc"] == d and ev in c["text"] for c in ch) for d, ev in EVIDENCE.items()}
    return ch, dict(strategy=strategy, chunks=len(ch), avg_words=round(float(np.mean(sizes)), 1),
                    min_words=min(sizes), max_words=max(sizes),
                    duplicate_words=sum(sizes) - orig, survive=surv)


STRATEGY_NAME = {"0": "window-40/overlap-0", "10": "window-40/overlap-10", "sentence": "sentence-40"}
CHUNK_STATS = {}


def task2():
    H("TASK 2 (Intermediate) - chunk comparison on the 7-document corpus")
    chunk_self_tests()
    print()
    for s in ["0", "10", "sentence"]:
        ch, st = chunk_stats(DOCS, s)
        CHUNK_STATS[s] = st
        for c in ch:
            assert c["text"] == DOCS[c["doc"]][c["start"]:c["end"]] and c["text"].strip()
        print(f"{STRATEGY_NAME[s]:22s} chunks={st['chunks']:3d} avg={st['avg_words']:5.1f} "
              f"min={st['min_words']:2d} max={st['max_words']:2d} duplicate_words={st['duplicate_words']:3d}")
        print("   evidence kept whole in one chunk:", st["survive"])
    print("\nExample chunk records (id, doc, start, end) for 'late' with sentence packing:")
    for c in [c for c in make_chunks(DOCS, "sentence") if c["doc"] == "late"]:
        print("  ", c["id"], c["start"], c["end"], "|", c["text"][:70], "...")

    H("TASK 2 - semantic suggestions on chunk records (3 paraphrased queries)")
    records = make_chunks(DOCS, "sentence")
    paraphrases = ["How can I regain access to my account?",
                   "What if I was sick and handed my work in late?",
                   "Is it okay for a freelancer to work from home?"]
    engines = {"tfidf": Retriever(records, "tfidf")}
    if get_model() is not None:
        engines["embedding"] = Retriever(records, "embedding")
    for q in paraphrases:
        print(f"\nQuery: {q!r}")
        for name, eng in engines.items():
            print(f"  [{name}]")
            for r, s in eng.search(q, k=3):
                print(f"    {r['id']:20s} score={s:.3f} chars[{r['start']}:{r['end']}] {r['text'][:80]}...")
        if "embedding" not in engines:
            print("  [embedding skipped: encoder unavailable]")
    if get_model() is not None:
        print("\n(Reference wiring check, task2_semantic_check)")
        recs = make_chunks({"it": "A forgotten password can be reset using the recovery link.",
                            "assessment": "Assignments are uploaded through the learning portal."}, "sentence")
        for backend in ["tfidf", "embedding"]:
            print("Backend:", backend)
            show_hits(Retriever(recs, backend).search("How can I regain access to my account?", k=2))


# =============================================================================
# TASK 3 - evaluation of 6 configurations
# =============================================================================
RESULTS, LOGS, CONFIG_STATE = [], [], {}


def first_relevant_rank(row, chunks, q):
    top = np.argsort(-row, kind="stable")[:3]
    for rank, i in enumerate(top, 1):
        if chunks[i]["doc"] == q["doc"] and q["evidence"] in chunks[i]["text"]:
            return rank
    return None


def evaluate(scores, chunks, items):
    hits, rr = [], []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        rank = first_relevant_rank(row, chunks, q)
        hits.append(rank is not None)
        rr.append(1 / rank if rank else 0)
    return (float(np.mean(hits)), float(np.mean(rr))) if hits else (None, None)


def diagnose(q, chunks, row):
    """Heuristic explanation of why a query failed (used for the failure report)."""
    top = chunks[int(np.argsort(-row, kind="stable")[0])]
    if not any(c["doc"] == q["doc"] and q["evidence"] in c["text"] for c in chunks):
        return "evidence split across chunk boundary (no single chunk holds the full rule + condition)", top
    if top["doc"] != q["doc"]:
        return f"related/conflicting or lexically similar document '{top['doc']}' outranked the gold document", top
    return "right document but a neighbouring/distractor chunk outranked the evidence chunk", top


def task3():
    H("TASK 3 (Advanced) - Evaluate document retrieval (3 chunkers x 2 retrievers)")
    model = get_model()
    print("Model load seconds (measured separately):",
          f"{MODEL_LOAD_SECONDS:.2f}" if MODEL_LOAD_SECONDS else "n/a")
    query_texts = [q["query"] for q in QUERIES]
    methods = ["tfidf"] + (["embedding"] if model is not None else [])
    for strategy in ["0", "10", "sentence"]:
        chunks = make_chunks(DOCS, strategy)
        texts = [c["text"] for c in chunks]
        max_tok = None
        if model is not None:   # audit token lengths
            lengths = [len(model.tokenizer(t, truncation=False)["input_ids"]) for t in texts + query_texts]
            max_tok = max(lengths)
            print(f"[{STRATEGY_NAME[strategy]}] token audit: max={max_tok} limit={model.max_seq_length} "
                  f"over-limit={sum(l > model.max_seq_length for l in lengths)}")
            if max_tok > model.max_seq_length:
                raise ValueError("Model limit exceeded; rechunk or shorten query")
        for method in methods:
            t = time.perf_counter()
            if method == "tfidf":
                vec = TfidfVectorizer()
                C = vec.fit_transform(texts)
                Q = vec.transform(query_texts)
            else:
                C = model.encode(texts, normalize_embeddings=True)
                Q = model.encode(query_texts, normalize_embeddings=True)
            enc = time.perf_counter() - t
            t = time.perf_counter()
            scores = (Q @ C.T).toarray() if method == "tfidf" else Q @ C.T
            srch = time.perf_counter() - t
            CONFIG_STATE[(strategy, method)] = dict(scores=scores, chunks=chunks)
            for split in ["dev", "test"]:
                idx = [i for i, q in enumerate(QUERIES) if q["split"] == split]
                h, m = evaluate(scores[idx], chunks, [QUERIES[i] for i in idx])
                RESULTS.append(dict(strategy=STRATEGY_NAME[strategy], skey=strategy, method=method,
                                    split=split, hit=h, mrr=m, chunks=len(chunks),
                                    encode_s=enc, search_s=srch, max_tokens=max_tok))
                print(f"{STRATEGY_NAME[strategy]:22s} {method:9s} {split:4s} Hit@3={h:.3f} MRR@3={m:.3f} "
                      f"chunks={len(chunks):3d} encode={enc:.3f}s search={srch:.4f}s")
            for row, q in zip(scores, QUERIES):
                for rank, i in enumerate(np.argsort(-row, kind="stable")[:3], 1):
                    c = chunks[i]
                    LOGS.append(dict(strategy=STRATEGY_NAME[strategy], method=method, query=q["query"],
                                     split=q["split"], answerable=q["doc"] is not None, rank=rank,
                                     score=round(float(row[i]), 4), chunk_id=c["id"], doc=c["doc"],
                                     start=c["start"], end=c["end"],
                                     relevant=bool(q["doc"] is not None and c["doc"] == q["doc"]
                                                   and q["evidence"] in c["text"]),
                                     text=c["text"]))
    with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(LOGS[0]))
        w.writeheader()
        w.writerows(LOGS)
    print("\nWrote retrieval_log.csv with", len(LOGS), "rows")

    # ---- manual metric arithmetic check on one top-3 list
    key = ("sentence", methods[-1])
    st = CONFIG_STATE[key]
    qi = next(i for i, q in enumerate(QUERIES) if q["doc"] is not None)
    top = np.argsort(-st["scores"][qi], kind="stable")[:3]
    print(f"\nManual check [{key}] query={QUERIES[qi]['query']!r}")
    for rank, i in enumerate(top, 1):
        c = st["chunks"][i]
        ok = c["doc"] == QUERIES[qi]["doc"] and QUERIES[qi]["evidence"] in c["text"]
        print(f"  rank {rank}: {c['id']} relevant={ok}")

    # ---- strategy selection on DEV only
    dev = [r for r in RESULTS if r["split"] == "dev"]
    dev.sort(key=lambda r: (-r["mrr"], -r["hit"], r["chunks"]))
    best = dev[0]
    sel = (best["skey"], best["method"])
    print(f"\nSELECTED on development data only: {STRATEGY_NAME[sel[0]]} + {sel[1]} "
          f"(dev Hit@3={best['hit']:.3f}, MRR@3={best['mrr']:.3f})")
    t = next(r for r in RESULTS if (r["skey"], r["method"], r["split"]) == (sel[0], sel[1], "test"))
    print(f"HELD-OUT TEST (no retuning): Hit@3={t['hit']:.3f}, MRR@3={t['mrr']:.3f}")
    print("Caveat: 4 dev + 4 test answerable queries = tiny sample; differences of one query are not significant.")

    # ---- failures
    H("TASK 3 - failures / edge cases")
    failures, seen = [], set()
    order = [sel] + [k for k in CONFIG_STATE if k != sel]
    for k in order:
        st = CONFIG_STATE[k]
        for row, q in zip(st["scores"], QUERIES):
            if q["doc"] is None:
                continue
            rank = first_relevant_rank(row, st["chunks"], q)
            if rank != 1:
                cause, topc = diagnose(q, st["chunks"], row)
                key2 = (q["query"], cause)
                if key2 in seen:
                    continue
                seen.add(key2)
                failures.append(dict(config=f"{STRATEGY_NAME[k[0]]} + {k[1]}", query=q["query"],
                                     rank=rank, top_chunk=topc["id"], top_text=topc["text"][:160],
                                     cause=cause))
    for i, f in enumerate(failures[:8], 1):
        print(f"\nFailure {i}: [{f['config']}] {f['query']!r}  first relevant rank={f['rank']}")
        print("   top-1 chunk:", f["top_chunk"], "|", f["top_text"], "...")
        print("   likely cause:", f["cause"])
    if len(failures) < 3:
        print("\n(Fewer than 3 automatic failures found - add harder queries or compare rank-2/3 cases;"
              " also use the conflicting-source and abbreviation examples below.)")
    print("\nEdge case (conflicting sources): 'What proof is needed when I was ill?' competes with the "
          "archived 2019 policy that says NO medical evidence is needed - high similarity != current/correct.")
    print("Edge case (abbreviations): 'approx.' / 'e.g.' in the printer guide trigger false sentence cuts.")
    print("Edge case (negation): the encoder ranks 'I do not need a password reset' near the password passage.")

    # ---- unanswerable queries
    H("TASK 3 - unanswerable queries")
    for k in order[:2] + [x for x in order[2:] if x[1] == "embedding"][:1]:
        st = CONFIG_STATE[k]
        for row, q in zip(st["scores"], QUERIES):
            if q["doc"] is None:
                i = int(np.argmax(row))
                print(f"[{STRATEGY_NAME[k[0]]} + {k[1]}] {q['query']!r} -> top-1 {st['chunks'][i]['id']} "
                      f"score={row[i]:.3f}")
    print("""WHY text is still returned: ranking only sorts candidates by similarity; arg-max always exists.
Dense cosine scores are rarely zero (unrelated text still shares some directions in the vector space);
TF-IDF only returns nothing when the query has zero vocabulary overlap. Neither measures
'is there an answer in the corpus'. A rejection threshold must be calibrated on DEV examples.""")

    # ---- threshold calibration on dev (selected config) - never tuned on test
    st = CONFIG_STATE[sel]
    top1 = [float(np.max(r)) for r in st["scores"]]
    dev_ans = [top1[i] for i, q in enumerate(QUERIES) if q["split"] == "dev" and q["doc"] is not None]
    dev_un = [top1[i] for i, q in enumerate(QUERIES) if q["split"] == "dev" and q["doc"] is None]
    threshold = None
    if dev_un and dev_ans and max(dev_un) < min(dev_ans):
        threshold = (max(dev_un) + min(dev_ans)) / 2
        print(f"\nDev-calibrated rejection threshold for {sel}: {threshold:.3f} "
              f"(max dev-unanswerable={max(dev_un):.3f} < min dev-answerable={min(dev_ans):.3f})")
        for i, q in enumerate(QUERIES):
            if q["split"] == "test":
                decision = "ACCEPT" if top1[i] >= threshold else "REJECT"
                truth = "answerable" if q["doc"] else "unanswerable"
                print(f"   test: {q['query']!r:48s} top1={top1[i]:.3f} {decision:6s} ({truth})")
    else:
        print("\nDev scores of answerable/unanswerable queries are not separable -> no threshold adopted; "
              "keep the confirmation-only workflow.")
    return sel, threshold, failures[:8]


# =============================================================================
# DOCUMENT CHATBOT (document_chatbot.py) - confirmation-based evidence display
# =============================================================================
DOC_TRANSCRIPT = []


def document_chatbot(engine, inputs=None, threshold=None, transcript=None):
    feed = iter(inputs) if inputs is not None else None

    def say(*a):
        s = " ".join(str(x) for x in a)
        print(s)
        if transcript is not None:
            transcript.append(s)

    def ask(prompt):
        if feed is None:
            v = input(prompt)
        else:
            try:
                v = next(feed)
            except StopIteration:
                raise EOFError
            print(prompt + v)
        if transcript is not None:
            transcript.append(prompt + v)
        return v

    say("Evidence assistant. Type exit to stop.")
    while True:
        try:
            query = ask("Question: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        if not query.strip():
            say("Please enter a question.")
            continue
        hits = engine.search(query)
        if hits and threshold is not None and hits[0][1] < threshold:
            hits = []      # below the dev-calibrated threshold
        if not hits:
            say("No candidate evidence; rephrase or contact a human.")
            continue
        say("Candidate evidence; relevance must be checked:")
        for number, (r, score) in enumerate(hits, 1):
            say(number, r["id"], f"{score:.3f}", "characters", r["start"], r["end"])
        chosen = ask("Choose 1-3 to inspect; Enter if none apply: ")
        if not chosen.isdigit() or not 1 <= int(chosen) <= len(hits):
            say("No answer selected. Rephrase or contact a human.")
            continue
        r, score = hits[int(chosen) - 1]
        say("Retrieved passage:", r["text"])
        say("Source:", r["doc"], "characters", r["start"], r["end"])
        say("(Candidate evidence only - not independently verified.)")


# =============================================================================
# REPORT (Lab6_Report.docx) - filled with the ACTUAL numbers produced above
# =============================================================================
def build_report(sel, threshold, failures):
    from docx import Document
    import sklearn
    d = Document()
    d.add_heading("Lab 6 Report - NLP Fundamentals: Tokenization, Embeddings & Text Chunking", 0)
    d.add_paragraph("Student name / ID: ____________________    CLO3 (C3: Apply)")

    d.add_heading("1. Scenario and objective", 1)
    d.add_paragraph("Scenario: university help desk search over handbook, IT and policy documents "
                    "(synthetic data). Objective: apply tokenization, embeddings and chunking to build a "
                    "traceable retrieval pipeline, compare behaviour and explain limitations.")
    d.add_heading("2. Dataset / source", 1)
    d.add_paragraph(f"{len(DOCS)} synthetic documents (150-250 words each), written for teaching; they are not real "
                    f"policies. Includes related but conflicting passages (current vs archived assessment policy; "
                    f"contractor vs employee remote-work rules). {sum(q['doc'] is not None for q in QUERIES)} "
                    f"answerable queries (4 dev, 4 test) and 2 unanswerable queries. Evidence strings are exact "
                    f"substrings labelled before inspecting rankings.")
    d.add_heading("3. Tokenization choices", 1)
    d.add_paragraph("Compared str.split(), Unicode regex \\b\\w+\\b and the model's sub-word tokenizer. Originals are "
                    "kept; no lowercasing/stripping is applied to stored text. The regex is not a complete "
                    "multilingual tokenizer.")
    t = d.add_table(rows=1, cols=5)
    t.style = "Table Grid"
    for i, h in enumerate(["Input", "Whitespace (n)", "Word-like regex (n)", "Model sub-words", "Note"]):
        t.rows[0].cells[i].text = h
    for r in TOKEN_ROWS:
        c = t.add_row().cells
        c[0].text = repr(r["text"])
        c[1].text = f"{r['ws']} ({len(r['ws'])})"
        c[2].text = f"{r['words']} ({len(r['words'])})"
        c[3].text = str(r["sub"]) if r["sub"] else "n/a"
        c[4].text = ""
    d.add_paragraph("Differences: punctuation stays attached with split() but is dropped by the regex; contractions "
                    "and e-mail addresses are split by the regex. Normalization risk: removing punctuation or "
                    "lowercasing destroys negation and case-sensitive identifiers.")

    d.add_heading("4. Model, versions and chunk settings", 1)
    try:
        import sentence_transformers
        stv = sentence_transformers.__version__
    except Exception:
        stv = "not available"
    d.add_paragraph(f"Encoder: {MODEL_NAME} (CPU), max_seq_length=256 word pieces. sentence-transformers {stv}, "
                    f"scikit-learn {sklearn.__version__}, numpy {np.__version__}. Chunk sizes are measured in "
                    f"whitespace words: windows of 40 words with overlap 0 and 10; sentence-aware packing with a "
                    f"40-word budget (long sentences fall back to 40-word windows). Model load time (measured "
                    f"separately): {MODEL_LOAD_SECONDS if MODEL_LOAD_SECONDS else 'n/a'} s.")
    t = d.add_table(rows=1, cols=7)
    t.style = "Table Grid"
    for i, h in enumerate(["Strategy", "Chunks", "Avg words", "Min", "Max", "Duplicate words", "Evidence kept whole"]):
        t.rows[0].cells[i].text = h
    for s, st in CHUNK_STATS.items():
        c = t.add_row().cells
        c[0].text, c[1].text, c[2].text = STRATEGY_NAME[s], str(st["chunks"]), str(st["avg_words"])
        c[3].text, c[4].text, c[5].text = str(st["min_words"]), str(st["max_words"]), str(st["duplicate_words"])
        c[6].text = f"{sum(st['survive'].values())}/{len(st['survive'])}"

    d.add_heading("5. Retrieval results (all configurations)", 1)
    d.add_paragraph("Timing method: time.perf_counter(); encode = TF-IDF fit+transform or encoder.encode on all chunks "
                    "and queries; search = matrix product; model load measured once separately.")
    t = d.add_table(rows=1, cols=8)
    t.style = "Table Grid"
    for i, h in enumerate(["Strategy", "Retriever", "Split", "Hit@3", "MRR@3", "Chunks", "Encode s", "Search s"]):
        t.rows[0].cells[i].text = h
    for r in RESULTS:
        c = t.add_row().cells
        vals = [r["strategy"], r["method"], r["split"], f"{r['hit']:.3f}", f"{r['mrr']:.3f}",
                str(r["chunks"]), f"{r['encode_s']:.3f}", f"{r['search_s']:.4f}"]
        for i, v in enumerate(vals):
            c[i].text = v
    d.add_heading("6. Selected strategy and justification", 1)
    d.add_paragraph(f"Selected on DEVELOPMENT results only (highest MRR@3, ties by Hit@3 then fewer chunks): "
                    f"{STRATEGY_NAME[sel[0]]} + {sel[1]}. Held-out test results were then reported without retuning "
                    f"(table above). With 4 dev and 4 test queries, differences of a single query are not "
                    f"statistically meaningful.")
    if threshold is not None:
        d.add_paragraph(f"A rejection threshold of {threshold:.3f} was calibrated on dev examples only; it is not "
                        f"a universal cosine cutoff.")
    d.add_heading("7. Chatbot transcripts", 1)
    d.add_paragraph("FAQ chatbot (TF-IDF) - exact answer, suggestion, empty and unsupported queries:")
    for label, q, a in FAQ_TRANSCRIPT:
        d.add_paragraph(f"[{label}]\nYou: {q}\nBot: {a}")
    d.add_paragraph("Document chatbot (source-traceable evidence, confirmation workflow):")
    d.add_paragraph("\n".join(DOC_TRANSCRIPT))
    d.add_paragraph("Confirmation/fallback behaviour: only exact canonical FAQ questions are answered automatically; "
                    "for documents the bot shows candidate IDs/offsets and displays a passage only after the user "
                    "chooses it. Empty input, no candidates (or scores below the calibrated threshold) and an "
                    "invalid choice all produce a fallback message.")
    d.add_heading("8. Failures and edge cases", 1)
    for i, f in enumerate(failures[:5], 1):
        d.add_paragraph(f"Failure {i} [{f['config']}] query: {f['query']!r}; first relevant rank: {f['rank']}. "
                        f"Retrieved: {f['top_chunk']} - {f['top_text']}... Likely cause: {f['cause']}. "
                        f"Improvement: larger/semantic-aware chunks, metadata (version/date) filters, hybrid "
                        f"TF-IDF + embedding search, reranking.")
    d.add_paragraph("Why highest-score retrieval still returns text for an unanswerable query: ranking always has a "
                    "maximum; scores measure similarity, not answerability. Calibrate a threshold on dev data.")
    d.add_paragraph("FAQ matching vs document retrieval: FAQs are short, curated and self-contained (missing context "
                    "is rare), whereas document chunks can omit the exception or conflict with another version.")
    d.add_heading("9. Limitations", 1)
    d.add_paragraph("Tiny synthetic corpus; English-only encoder; simple regex sentence splitter; word-count budgets "
                    "are not token counts; retrieved text is evidence to inspect, not a verified answer.")
    d.add_heading("10. Reproduction steps", 1)
    d.add_paragraph("python -m venv .venv; install numpy scikit-learn sentence-transformers python-docx; "
                    "run lab6_all_tasks_colab.py (or the split task files). Outputs: corpus.json, queries.json, "
                    "retrieval_log.csv, Lab6_Report.docx.")
    d.add_heading("11. Reflection (write your OWN answers)", 1)
    for qn in ["What did you learn today?", "How will you apply this in a real AI system?",
               "What failed or surprised you?", "What would you do differently next time?"]:
        d.add_paragraph(qn, style="List Bullet")
        d.add_paragraph("Answer: ______________________________________________")
    d.save("Lab6_Report.docx")
    print("Wrote Lab6_Report.docx")


# =============================================================================
# MAIN
# =============================================================================
def main():
    t0 = time.perf_counter()
    write_dataset()
    get_model()                      # load once; failure is handled gracefully
    run_scenarios()
    guided_tokens()
    guided_embeddings()
    guided_limits()
    guided_metrics()
    task1_tokens()
    task1_chatbot()
    task2()                          # includes guided_chunks + boundary tests
    guided_suggestions()
    sel, threshold, failures = task3()

    H("DOCUMENT CHATBOT (scripted demo; set INTERACTIVE=True for live input)")
    method = sel[1] if (sel[1] == "embedding" and get_model() is not None) else "tfidf"
    engine = Retriever(make_chunks(DOCS, sel[0]), method)
    demo = ["Can contractors work remotely?", "1",
            "What are tomorrow's flight prices?", "",
            "What proof is needed when I was ill?", "1",
            "", "How can I recover my login access?", "", "exit"]
    document_chatbot(engine, inputs=demo, threshold=threshold, transcript=DOC_TRANSCRIPT)

    build_report(sel, threshold, failures)
    print("\nFILES in", os.getcwd(), ":", sorted(os.listdir(".")))
    print(f"Total runtime: {time.perf_counter() - t0:.1f}s")
    if INTERACTIVE:
        document_chatbot(engine, threshold=threshold)


if __name__ == "__main__":
    main()

Installing python-docx ...
Working folder: /content/lab6_nlp

DATASET - corpus.json / queries.json
late              186 words 
late_archived     165 words 
password          187 words 
assignment        176 words 
contractor        169 words 
employee_remote   171 words 
printer           186 words 
Wrote corpus.json and queries.json; 8 answerable + 2 unanswerable queries


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[model] loaded sentence-transformers/all-MiniLM-L6-v2 in 4.25s, max_seq_length=256

SCENARIO 1 - University handbook search
assessment:late score=0.506
Late submission is normally penalized. Students with illness may request an extension with medical evidence before the deadline.

--- experiment S1 paraphrase: query='I was sick; can I hand it in tomorrow?' filters=None
  tfidf     -> no matches
  embedding -> [('assessment:late', 0.288), ('assessment:format', 0.123)]

SCENARIO 2 - Customer support knowledge base
returns:damage score=0.373
For a damaged parcel, contact support within 48 hours and attach photographs of the item and packaging.

--- experiment S2 NO filter: query='damaged parcel photographs' filters=None
  tfidf     -> [('returns:damage', 0.373), ('returns:damage:other', 0.252), ('returns:change', 0.0)]
  embedding -> [('returns:damage', 0.74), ('returns:damage:other', 0.495), ('returns:change', 0.219)]

--- experiment S2 paraphrase: query='My parcel arrived broken' filter

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (302 > 256). Running this sequence through the model will result in indexing errors



Query [paraphrase]: 'How can I recover my login?'   embedding shape: (3, 384)
  0 0.627 Reset a forgotten password using the recovery link.
  1 0.072 Submit assignments through the learning portal.
  2 0.018 Check ink levels when a printer produces blank pages.

Query [NEGATED (experiment)]: 'I do not need a password reset'   embedding shape: (3, 384)
  0 0.713 Reset a forgotten password using the recovery link.
  2 0.151 Check ink levels when a printer produces blank pages.
  1 0.1 Submit assignments through the learning portal.

Observation: the negated query still favours the password passage - the encoder
captures topic similarity, not logical negation. Score != correctness.

6.4 guided_limits.py
short tokens: 6 limit: 256 fits: True
long tokens: 302 limit: 256 fits: False
'CS4944': whitespace=1 model_tokens=5 (incl. special tokens)
'help@example.com': whitespace=1 model_tokens=7 (incl. special tokens)
'POST /v2/token/refresh': whitespace=2 model_tokens=11 (incl. special tokens)
'